# 🥇 Gold Trend Classification using Machine Learning

Welcome to this end-to-end Quantitative Machine Learning project. The objective is to predict whether Gold Futures (`GC=F`) will close **UP** or **DOWN** on the next trading day using historical financial market data and engineered technical indicators.

This notebook demonstrates a complete financial ML workflow with a strict emphasis on preventing data leakage and look-ahead bias.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# 🎨 STRICT GLOBAL VISUALIZATION SYSTEM
plt.style.use("dark_background")
sns.set_theme(style="dark")
plt.rcParams.update({
    'figure.facecolor': '#0E1117',
    'axes.facecolor': '#0E1117',
    'text.color': 'white',
    'axes.labelcolor': 'lightgray',
    'xtick.color': 'lightgray',
    'ytick.color': 'lightgray',
    'axes.edgecolor': '#333333'
})

COLORS = {
    'blue': '#4FC3F7',
    'orange': '#FFB74D',
    'green': '#81C784',
    'red': '#E57373'
}

## 1. Data Ingestion
We download over 15 years of daily historical data for Gold Futures using `yfinance` to ensure a robust dataset.

In [ ]:
ticker = 'GC=F'
df = yf.download(ticker, start='2008-01-01', end='2024-01-01')

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.dropna(inplace=True)
print(f"Raw dataset shape: {df.shape}")
df.head()

## 2. Technical Indicator Engineering
**STRICT RULE:** We calculate all technical indicators on the *continuous* dataset.

In [ ]:
def add_indicators(data):
    df = data.copy()
    df['SMA_7'] = df['Close'].rolling(window=7).mean()
    df['SMA_30'] = df['Close'].rolling(window=30).mean()
    
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    
    df['Daily_Return'] = df['Close'].pct_change()
    df['Volatility_14'] = df['Daily_Return'].rolling(window=14).std()
    return df

df_indicators = add_indicators(df)
df_indicators.dropna(inplace=True)
print(f"Dataset shape after indicators: {df_indicators.shape}")

## 3. Target Creation & NEUTRAL Filtering
We create a target based on the *next day's* return (t+1).

In [ ]:
threshold = 0.003 # 0.3%
df_indicators['Next_Return'] = df_indicators['Close'].pct_change().shift(-1)

conditions = [
    (df_indicators['Next_Return'] > threshold),
    (df_indicators['Next_Return'] < -threshold)
]
choices = [1, 0]
df_indicators['Target'] = np.select(conditions, choices, default=-1)

print(f"Rows before filtering: {len(df_indicators)}")
df_final = df_indicators[df_indicators['Target'] != -1].copy()
print(f"Rows after filtering NEUTRAL: {len(df_final)}")
df_final.drop(columns=['Next_Return'], inplace=True)

## 4. Chronological Split & Leakage-Free Scaling
**STRICT RULE:** Chronological split (no shuffle).

In [ ]:
split_index = int(len(df_final) * 0.8)
train, test = df_final.iloc[:split_index], df_final.iloc[split_index:]

features = ['SMA_7', 'SMA_30', 'RSI_14', 'Daily_Return', 'Volatility_14']
X_train, y_train = train[features], train['Target']
X_test, y_test = test[features], test['Target']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training & Evaluation

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
preds = model.predict(X_test_scaled)

print("--- Classification Report ---")
print(classification_report(y_test, preds, target_names=['DOWN', 'UP']))

### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'], annot_kws={"color": "white"})
plt.title('Confusion Matrix', color='white')
plt.ylabel('True Label', color='lightgray')
plt.xlabel('Predicted Label', color='lightgray')
plt.show()